In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import torch 
import os
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms
import pandas as pd
from sklearn.model_selection import train_test_split
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import fastparquet
from sklearn.metrics import classification_report




In [ ]:
SEED = 42
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Verwendetes Gerät:", device)

Verwendetes Gerät: cuda


In [ ]:
dx=pd.read_csv("Data/Train_set.csv")
label_to_id = pd.read_csv("Data/label_to_id_mapping.csv").set_index("label")["id"].to_dict()

dx.head()


,X,y
0,hagrid-sample-30k-384p/hagrid_30k/train_val_fo...,3
1,hagrid-sample-30k-384p/hagrid_30k/train_val_fo...,3
2,hagrid-sample-30k-384p/hagrid_30k/train_val_ca...,0
3,hagrid-sample-30k-384p/hagrid_30k/train_val_li...,4
4,hagrid-sample-30k-384p/hagrid_30k/train_val_di...,1


In [ ]:

X_train, X_val, y_train, y_val = train_test_split(dx["X"], dx["y"], test_size=0.2, random_state=42)



In [ ]:
class CachedImageDataset(torch.utils.data.Dataset):
    def __init__(self, dt, dl, transform=None):
        self.data = []
        self.labels = dl
        self.dt = dt
        self.transform = transform


        for i in range(len(self.dt)):
            path = self.dt[i]
            img = Image.open(path).convert("RGB")
   
            label = torch.tensor(self.labels[i], dtype=torch.long)
            self.data.append((img, label))

            

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img, label = self.data[index]

        if self.transform:
            img = self.transform(img)

        return img, label



In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomAffine(
        degrees=10,
        translate=(0.05, 0.05),
        scale=(0.9, 1.1),
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.03,
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5],
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5],
    ),
])





train_dataset = CachedImageDataset(
    X_train.tolist(),
    y_train.tolist(),
    transform=train_transform
)

val_dataset = CachedImageDataset(
    X_val.tolist(),
    y_val.tolist(),
    transform=val_transform
)

loader_options = {
    "batch_size": 256,
    "num_workers": 0,
    "pin_memory": True
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_options
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_options
)

In [ ]:

MODEL_PATH = "models/deit-tiny"
model = AutoModelForImageClassification.from_pretrained(
    "models/deit-tiny",
    num_labels=len(label_to_id),
    ignore_mismatched_sizes=True
)

model.to(device)



[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: models/deit-tiny
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([5])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 192]) vs model:torch.Size([5, 192])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=192, out_features=192, bias=True)
          (k_proj): Linear(in_features=192, out_features=192, bias=True)
          (v_proj): Linear(in_features=192, out_features=192, bias=True)
          (o_proj): Linear(in_features=192, out_features=192, bias=True)
        )
        (layernorm_before): LayerNorm((192,), eps=1e-12, elementwise_affine=True)
        (layernorm_after): LayerNorm((192,), eps=1e-12, elementwise_affine=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=192, out_features=768, bias=True)
          (fc2): Linear(in_features=768, out_f

In [ ]:



optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
epochs = 10
best_val_accuracy = 0
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        output = model(
            pixel_values=images,
            labels=labels
        )

        loss = output.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoche {epoch + 1}/{epochs}"
        f"Loss: {total_loss / len(train_loader):.4f}"
    )
    model.eval()
    total_correct = 0
    total_samples = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model(pixel_values=images)
            predictions = output.logits.argmax(dim=1)
            total_correct += (predictions == labels).sum().item()
            total_samples += labels.size(0)
    accuracy = total_correct / total_samples
    print(f"Validation Accuracy: {accuracy:.4f}")
    if accuracy > best_val_accuracy:
        best_val_accuracy = accuracy
        model.save_pretrained("best_model")
        print(f"Neues bestes Modell gespeichert: {best_val_accuracy:.4f}")



Epoche 1/10Loss: 1.6127
Validation Accuracy: 0.3067


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Neues bestes Modell gespeichert: 0.3067
Epoche 2/10Loss: 1.3755
Validation Accuracy: 0.6378


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Neues bestes Modell gespeichert: 0.6378
Epoche 3/10Loss: 0.8719
Validation Accuracy: 0.7789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Neues bestes Modell gespeichert: 0.8667
Epoche 9/10Loss: 0.2893
Validation Accuracy: 0.8833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Neues bestes Modell gespeichert: 0.8833
Epoche 10/10Loss: 0.2617
Validation Accuracy: 0.8756
